# PCFI261 - Solemne 3
## Notebook 02: red neuronal libre para theta(t)

Este notebook entrena una red densa con `tensorflow.keras` para aproximar la senal angular medida del pendulo:

$$\theta_{NN}(t) \approx \theta(t).$$

La red libre minimiza solo error contra datos. Por lo tanto, un buen ajuste numerico no implica por si solo que la red respete la ecuacion diferencial del pendulo.

> **Uso esperado.** Esta es una plantilla de trabajo. No debe entregarse sin completar los valores marcados como `TODO`, sin revisar las figuras, ni sin discutir las decisiones experimentales y fisicas en el informe.

## 0. Preparacion

Instale, si es necesario:

```bash
pip install numpy pandas matplotlib scikit-learn tensorflow
```

Ejecute primero el notebook 01 para generar `salidas_pendulo/trayectoria_pendulo.csv`.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True

tf.keras.utils.set_random_seed(123)

In [ ]:
# ============================
# CONFIGURACION DEL ESTUDIANTE
# ============================

CSV_PATH = Path("salidas_pendulo/trayectoria_pendulo.csv")
OUTPUT_DIR = Path("salidas_pendulo")
OUTPUT_DIR.mkdir(exist_ok=True)

# Si el CSV no contiene theta, complete el pivote y active la reconstruccion.
PIVOTE_X = None
PIVOTE_Y = None

# Entrenamiento
TEST_FRACTION = 0.25
VALIDATION_MODE = "intercalada"  # opciones: "intercalada" o "aleatoria"
EPOCHS = 1000
LEARNING_RATE = 1e-2
BATCH_SIZE = None  # None usa todos los datos por epoca

## 1. Carga y revision de datos


In [ ]:
df = pd.read_csv(CSV_PATH)
df = df.replace([np.inf, -np.inf], np.nan)

if "t_rel" not in df.columns:
    df["t_rel"] = df["t"] - df["t"].min()

if "theta" not in df.columns:
    if PIVOTE_X is None or PIVOTE_Y is None:
        raise ValueError("El CSV no contiene theta. Complete PIVOTE_X y PIVOTE_Y para reconstruirla.")
    theta_raw = np.arctan2(df["x"].to_numpy() - PIVOTE_X, PIVOTE_Y - df["y"].to_numpy())
    df["theta"] = np.unwrap(theta_raw)
    df["theta"] = df["theta"] - np.nanmedian(df["theta"])

df = df.dropna(subset=["t_rel", "theta"]).sort_values("t_rel").reset_index(drop=True)

print(df.head())
print("N =", len(df))
print("amplitud angular maxima aproximada [rad] =", np.max(np.abs(df["theta"].to_numpy())))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
if {"x", "y"}.issubset(df.columns):
    axes[0].plot(df["x"], df["y"], ".")
    axes[0].invert_yaxis()
    axes[0].set_xlabel("x [pixeles]")
    axes[0].set_ylabel("y [pixeles]")
    axes[0].set_title("trayectoria extraida")
else:
    axes[0].axis("off")

axes[1].plot(df["t_rel"], df["theta"], ".-")
axes[1].set_xlabel("t_rel [s]")
axes[1].set_ylabel("theta [rad]")
axes[1].set_title("senal angular")
fig.tight_layout()

## 2. Limpieza minima y separacion entrenamiento/validacion

Una validacion intercalada evalua interpolacion dentro del intervalo observado. Una separacion aleatoria tambien es aceptable si se justifica. Para extrapolacion temporal, use un bloque final como validacion y expliquelo.


In [ ]:
# TODO: si detecta outliers, implemente y justifique aqui su criterio.
# Por defecto solo se ordenan los datos y se mantienen todos los puntos validos.

t_all = df[["t_rel"]].to_numpy(dtype=np.float32)
th_all = df[["theta"]].to_numpy(dtype=np.float32)

n = len(df)
if VALIDATION_MODE == "intercalada":
    step = max(int(round(1 / TEST_FRACTION)), 2)
    val_mask = (np.arange(n) % step) == 0
elif VALIDATION_MODE == "aleatoria":
    rng = np.random.default_rng(123)
    val_mask = rng.random(n) < TEST_FRACTION
else:
    raise ValueError("VALIDATION_MODE debe ser 'intercalada' o 'aleatoria'")

train_mask = ~val_mask

t_train = t_all[train_mask]
th_train = th_all[train_mask]
t_val = t_all[val_mask]
th_val = th_all[val_mask]

print("N train =", len(t_train), "N val =", len(t_val))

## 3. Normalizacion

La red se entrena mejor si la entrada y la salida tienen escala comparable. Se guardan las medias y desviaciones para volver a unidades fisicas al graficar.


In [ ]:
t_mean = t_train.mean(axis=0, keepdims=True)
t_std = t_train.std(axis=0, keepdims=True)
th_mean = th_train.mean(axis=0, keepdims=True)
th_std = th_train.std(axis=0, keepdims=True)

t_std[t_std == 0] = 1.0
th_std[th_std == 0] = 1.0

def normalizar_t(t):
    return (t - t_mean) / t_std

def normalizar_theta(theta):
    return (theta - th_mean) / th_std

def desnormalizar_theta(theta_norm):
    return theta_norm * th_std + th_mean

zt_train = normalizar_t(t_train)
zt_val = normalizar_t(t_val)
zth_train = normalizar_theta(th_train)
zth_val = normalizar_theta(th_val)

## 4. Modelo de red neuronal libre


In [ ]:
def construir_red_libre(width=32, depth=2, activation="tanh"):
    model = keras.Sequential(name="red_libre_theta")
    model.add(layers.Input(shape=(1,)))
    for _ in range(depth):
        model.add(layers.Dense(width, activation=activation))
    model.add(layers.Dense(1))
    return model

model = construir_red_libre(width=32, depth=2, activation="tanh")
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss="mse",
    metrics=[keras.metrics.MeanAbsoluteError(name="mae")],
)
model.summary()

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=100,
        restore_best_weights=True,
    )
]

fit_kwargs = {}
if BATCH_SIZE is not None:
    fit_kwargs["batch_size"] = BATCH_SIZE

history = model.fit(
    zt_train, zth_train,
    validation_data=(zt_val, zth_val),
    epochs=EPOCHS,
    verbose=0,
    callbacks=callbacks,
    **fit_kwargs,
)

print("epocas ejecutadas:", len(history.history["loss"]))
print("loss final:", history.history["loss"][-1])
print("val_loss final:", history.history["val_loss"][-1])

In [ ]:
plt.figure(figsize=(7, 4))
plt.semilogy(history.history["loss"], label="train")
plt.semilogy(history.history["val_loss"], label="validacion")
plt.xlabel("epoca")
plt.ylabel("MSE normalizado")
plt.legend()
plt.tight_layout()

## 5. Prediccion y metricas en unidades originales


In [ ]:
t_grid = np.linspace(t_all.min(), t_all.max(), 600, dtype=np.float32).reshape(-1, 1)
z_grid = normalizar_t(t_grid)
th_grid_pred = desnormalizar_theta(model.predict(z_grid, verbose=0))

th_val_pred = desnormalizar_theta(model.predict(zt_val, verbose=0))
mse_val = mean_squared_error(th_val, th_val_pred)
mae_val = mean_absolute_error(th_val, th_val_pred)

print(f"MSE validacion = {mse_val:.6e} rad^2")
print(f"MAE validacion = {mae_val:.6e} rad")

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(t_train[:, 0], th_train[:, 0], ".", label="train")
plt.plot(t_val[:, 0], th_val[:, 0], ".", label="validacion")
plt.plot(t_grid[:, 0], th_grid_pred[:, 0], "-", label="red libre")
plt.xlabel("t_rel [s]")
plt.ylabel("theta [rad]")
plt.legend()
plt.tight_layout()

In [ ]:
res_val = th_val_pred[:, 0] - th_val[:, 0]

plt.figure(figsize=(7, 3.5))
plt.axhline(0.0, linewidth=1)
plt.plot(t_val[:, 0], res_val, ".")
plt.xlabel("t_rel [s]")
plt.ylabel("residuo de datos [rad]")
plt.tight_layout()

## 6. Guardado de predicciones


In [ ]:
pred_df = pd.DataFrame({
    "t_rel": t_grid[:, 0],
    "theta_red_libre": th_grid_pred[:, 0],
})
pred_path = OUTPUT_DIR / "predicciones_red_libre.csv"
pred_df.to_csv(pred_path, index=False)
print(f"Predicciones guardadas en: {pred_path}")

fig_path = OUTPUT_DIR / "red_libre_theta.png"
plt.figure(figsize=(8, 4))
plt.plot(t_all[:, 0], th_all[:, 0], ".", label="datos")
plt.plot(t_grid[:, 0], th_grid_pred[:, 0], "-", label="red libre")
plt.xlabel("t_rel [s]")
plt.ylabel("theta [rad]")
plt.legend()
plt.tight_layout()
plt.savefig(fig_path, dpi=200)
print(f"Figura guardada en: {fig_path}")

## 7. Discusion minima para el informe

Responda con base en sus figuras:

1. La red, ¿interpola bien maximos y minimos?
2. ¿Suaviza demasiado la senal?
3. ¿Hay sobreajuste entre entrenamiento y validacion?
4. ¿Por que un MSE bajo no demuestra que la red aprendio la fisica?
5. ¿Que limitaciones experimentales afectan el ajuste?
